In [56]:
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [57]:
print(text[:200])
text = text[:200]
print('Length of text:', len(text))

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you
Length of text: 200


In [58]:
import torch
import numpy as np
from torch import nn

In [62]:
words = text.split()
# print(words[:10])
vocab = set(words)
vocab_size = len(vocab)

print(f'Vocab size: {vocab_size}')
# print(f'Vocab: {vocab}')

stoi = {word: i for i, word in enumerate(vocab)}
# print(stoi)
itos = {i: word for i, word in enumerate(vocab)}

encode = lambda x: [stoi[word] for word in x]
decode = lambda x: [itos[i] for i in x]

tokenIds = encode(words)
# print(tokenIds[:100])

print('Text:', words)
print('Token IDs:', tokenIds[:10])


Vocab size: 25
Text: ['First', 'Citizen:', 'Before', 'we', 'proceed', 'any', 'further,', 'hear', 'me', 'speak.', 'All:', 'Speak,', 'speak.', 'First', 'Citizen:', 'You', 'are', 'all', 'resolved', 'rather', 'to', 'die', 'than', 'to', 'famish?', 'All:', 'Resolved.', 'resolved.', 'First', 'Citizen:', 'First,', 'you']
Token IDs: [21, 4, 24, 1, 19, 2, 14, 23, 10, 22]


In [60]:
input_tensor = torch.tensor(tokenIds, dtype=torch.long)
print('Input tensor shape:', input_tensor.shape)
print('Input tensor:', input_tensor[:10])


Input tensor shape: torch.Size([32])
Input tensor: tensor([21,  4, 24,  1, 19,  2, 14, 23, 10, 22])


In [68]:
embedding_dim = 4
w = torch.randn(size=(vocab_size, embedding_dim), requires_grad=True)
print('Embedding matrix shape:', w.shape)
print('Embedding matrix:', w[:5])

Embedding matrix shape: torch.Size([25, 4])
Embedding matrix: tensor([[-0.1416, -1.0350, -0.3678, -1.0157],
        [ 0.4279, -1.5096,  0.7689, -1.2033],
        [-0.2002, -0.1465,  1.1305, -0.1874],
        [-0.5559, -1.0901, -1.8953,  1.7371],
        [-0.5015,  0.7227, -0.5825, -1.5516]], grad_fn=<SliceBackward0>)


In [69]:
output_manual_lookup = w[input_tensor]
print('Output shape (manual lookup):', output_manual_lookup.shape)
print('Output (manual lookup):', output_manual_lookup[:10])

Output shape (manual lookup): torch.Size([32, 4])
Output (manual lookup): tensor([[ 0.2898,  0.5700, -0.2124,  1.5121],
        [-0.5015,  0.7227, -0.5825, -1.5516],
        [-0.5134,  0.8113, -0.0106,  1.1826],
        [ 0.4279, -1.5096,  0.7689, -1.2033],
        [-0.7995, -0.4234,  0.3515,  1.4777],
        [-0.2002, -0.1465,  1.1305, -0.1874],
        [ 1.5409,  0.3846,  0.0349, -0.1872],
        [ 1.1339, -0.0106, -1.6618,  2.3653],
        [ 1.1993, -0.9906,  0.8926,  0.9440],
        [-0.1048,  2.5428, -1.2263,  0.0598]], grad_fn=<SliceBackward0>)


In [71]:
seq_len = input_tensor.shape[0]
print('Sequence length:', seq_len)
d_model = embedding_dim

Sequence length: 32


In [75]:
pe = torch.zeros(seq_len, d_model)
print('Positional encoding shape:', pe.shape)
position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
print('Position shape:', position.shape)
print('Position:', position[:10])
div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(np.log(10000.0) / d_model))
print('Div term shape:', div_term.shape)
print('Div term:', div_term)
pe[:, 0::2] = torch.sin(position * div_term)
pe[:, 1::2] = torch.cos(position * div_term)
print('Positional encoding:', pe[:10])

Positional encoding shape: torch.Size([32, 4])
Position shape: torch.Size([32, 1])
Position: tensor([[0.],
        [1.],
        [2.],
        [3.],
        [4.],
        [5.],
        [6.],
        [7.],
        [8.],
        [9.]])
Div term shape: torch.Size([2])
Div term: tensor([1.0000, 0.0100])
Positional encoding: tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  0.9999],
        [ 0.9093, -0.4161,  0.0200,  0.9998],
        [ 0.1411, -0.9900,  0.0300,  0.9996],
        [-0.7568, -0.6536,  0.0400,  0.9992],
        [-0.9589,  0.2837,  0.0500,  0.9988],
        [-0.2794,  0.9602,  0.0600,  0.9982],
        [ 0.6570,  0.7539,  0.0699,  0.9976],
        [ 0.9894, -0.1455,  0.0799,  0.9968],
        [ 0.4121, -0.9111,  0.0899,  0.9960]])


In [76]:
scaled_word_embeddings = output_manual_lookup * np.sqrt(d_model)
print('Scaled word embeddings shape:', scaled_word_embeddings.shape)
print('Scaled word embeddings:', scaled_word_embeddings[:10])

Scaled word embeddings shape: torch.Size([32, 4])
Scaled word embeddings: tensor([[ 0.5795,  1.1401, -0.4249,  3.0243],
        [-1.0030,  1.4454, -1.1650, -3.1032],
        [-1.0267,  1.6226, -0.0211,  2.3652],
        [ 0.8557, -3.0193,  1.5377, -2.4066],
        [-1.5989, -0.8467,  0.7029,  2.9553],
        [-0.4004, -0.2931,  2.2609, -0.3748],
        [ 3.0819,  0.7693,  0.0698, -0.3743],
        [ 2.2679, -0.0213, -3.3236,  4.7306],
        [ 2.3986, -1.9813,  1.7852,  1.8879],
        [-0.2097,  5.0856, -2.4526,  0.1197]], grad_fn=<SliceBackward0>)


In [78]:
pe = pe.unsqueeze(0)
print('Positional encoding shape after unsqueeze:', pe.shape)
final_embeddings = scaled_word_embeddings + pe
print('Final embeddings shape:', final_embeddings.shape)
print('Final embeddings:', final_embeddings[:10])

Positional encoding shape after unsqueeze: torch.Size([1, 1, 32, 4])
Final embeddings shape: torch.Size([1, 1, 32, 4])
Final embeddings: tensor([[[[ 5.7950e-01,  2.1401e+00, -4.2487e-01,  4.0243e+00],
          [-1.6150e-01,  1.9857e+00, -1.1550e+00, -2.1033e+00],
          [-1.1743e-01,  1.2064e+00, -1.1215e-03,  3.3650e+00],
          [ 9.9684e-01, -4.0093e+00,  1.5677e+00, -1.4070e+00],
          [-2.3557e+00, -1.5004e+00,  7.4292e-01,  3.9545e+00],
          [-1.3593e+00, -9.4090e-03,  2.3109e+00,  6.2400e-01],
          [ 2.8025e+00,  1.7294e+00,  1.2977e-01,  6.2388e-01],
          [ 2.9248e+00,  7.3263e-01, -3.2537e+00,  5.7282e+00],
          [ 3.3879e+00, -2.1268e+00,  1.8651e+00,  2.8847e+00],
          [ 2.0247e-01,  4.1745e+00, -2.3627e+00,  1.1156e+00],
          [-8.7149e-01, -1.9425e+00, -4.3266e-01,  1.4230e+00],
          [-1.2403e+00,  1.0119e+00, -8.3695e-01,  3.5606e+00],
          [-7.4622e-01,  5.9295e+00, -2.3329e+00,  1.1125e+00],
          [ 9.9967e-01,  2.0475